The way to determine if the parameters for PSF resampler are correct is to check how the PSD before and after resampling compare.

This notebook explores the computation of spectra for both input and resampled.

I am selecting a square area in healpix (rotated by 45°).
A parent cell barely fits the swath, not ideal.
I start from a center pixel, that I enlarge using healpix_geo.nested.kth_neighbourhood.
If I put those cell ids in ring, they can be sorted (left to right, top ring to bottom ring) and then reordered to be wrapped in a 2D array.

In [ ]:
import datetime as dt
import logging
from dataclasses import dataclass, asdict
from os import path

import cartopy.crs as ccrs
import healpix_geo.ring
import healpix_geo.nested
import matplotlib.pyplot as plt
import numpy as np
import tol_colors as tc
import xarray as xr
from cdshealpix import to_ring, from_ring
from healpix_resample import PSFResampler
from healpix_resample.base import ResampleResults
from matplotlib.collections import PolyCollection
from mpl_toolkits.axes_grid1 import make_axes_locatable

from swot_healpix.analysis.spectra import FFTComputer, average_psd_azimuth
from swot_healpix.analysis.spectra.ww3.spectral_shift import (
    put_directions_in_order,
    transform_from_ftphi_to_fphi
)
from swot_healpix.data import SWOT
from swot_healpix.utils import format_latlon, get_cell_size, add_look_arrows, add_look_arrow

%matplotlib inline

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
level = 15
cell_size = get_cell_size(level)

# Definitions

## Reindex diamond to 2d

In [ ]:
def reindex_ring_to_2d(n_side: int) -> np.ndarray:
    """Return indices to reorder cells from ring order to 2d.

    This uses the fact that cells in the ring indexing scheme are arranged from left
    to right and top to bottom.

    To use those indices: Be sure to use 'ring' indexing scheme and that cells are
    sorted by id:
    ```
    cell_ids = to_ring(cell_ids, depth=level)
    sort_indices = np.argsort(cell_ids)
    cell_ids = cell_ids[sort_indices]
    cell_data = cell_data[sort_indices]
    ```
    Then to wrap data into a 2d array:
    ```
    cell_data_2d = np.zeros((2**level, 2**level))
    np.put(cell_data_2d, reindex_ring_to_2d(level), cell_data)
    ```

    Example (level 2)
    -----------------
    Left is ring-order, right is 2D (flattened) order.
    ```
              0                        15
           1     2                  14    11
        3     4     5            13    10     7
     6     7     8     9  =>  12     9     6     3
       10    11    12             8     5     2
          13    14                   4     1
             15                         0
    ```
    """
    indices = np.zeros(n_side**2, dtype="i8")
    # top half, excluding middle ring
    i_cell_start = 0
    n_cell_ring = 1
    for i_ring in range(n_side-1):
        ix = np.arange(n_side-1-i_ring, n_side)
        iy = ix[::-1]
        indices[i_cell_start : i_cell_start + n_cell_ring] = iy * n_side + ix

        i_cell_start += n_cell_ring
        n_cell_ring += 1

    for i_ring in range(n_side):
        ix = np.arange(0, n_side-i_ring)
        iy = ix[::-1]
        indices[i_cell_start : i_cell_start + n_cell_ring] = iy * n_side + ix

        i_cell_start += n_cell_ring
        n_cell_ring -= 1

    indices = np.asarray(indices)
    return indices

## Load data

In [ ]:
# Load data
def load(cycle: int, track: int, lat: float, side: str = "left"):
    """Load SWOT data."""
    swot = SWOT(cycle=cycle, track=track, version="PGC0")
    swot_file = swot.get_file()
    swath = xr.open_dataset(swot_file, group=side, chunks=-1)

    center_lat = swath.latitude.isel(num_pixels=swath.num_pixels.size // 2).values
    center_lat = np.sort(center_lat)  # just in case, just only modify slighty over land
    i_line = np.searchsorted(center_lat, lat)

    swath = swath.isel(num_lines=slice(i_line - 250, i_line + 250))

    if side == "left":
        swath = swath.isel(num_pixels=slice(None, None, -1))
    if swot.pass_type == "descending":
        swath = swath.isel(num_lines=slice(None, None, -1))

    swath.longitude.load()
    swath.latitude.load()
    swath.ssh_karin_2.load()

    swath = swath.dropna("num_lines", how="all", subset=["longitude"])
    swath = swath.dropna("num_pixels", how="all", subset=["longitude"])

    swath["ssh_karin_2"] -= swath.ssh_karin_2.mean()

    return swath

## Get diamond

In [ ]:
def get_diamond(
    swath: xr.Dataset, n_ring: int, center_i_line: int | None = None
) -> tuple[int, int, np.ndarray]:
    """Find cell ids for a diamond centered on the swath."""
    if center_i_line is None:
        center_i_line = swath.num_lines.size // 2
    center_i_pixel = swath.num_pixels.size // 2
    center_pixel = swath.isel(num_pixels=center_i_pixel, num_lines=center_i_line)
    center_id = healpix_geo.nested.lonlat_to_healpix(
        float(center_pixel.longitude),
        float(center_pixel.latitude),
        depth=level,
        ellipsoid="WGS84",
    )
    n_side = 2 * n_ring + 1
    return n_side, center_id, healpix_geo.nested.kth_neighbourhood(
        center_id, depth=level, ring=n_ring
    )[0]

## Resample and reshape

In [ ]:
def resample(
    swath: xr.Dataset,
    diamond_cell_ids: np.ndarray,
    lam: float = 0.0,
    verbose: float = True,
    **kwargs,
) -> ResampleResults:
    """Resample SSH."""
    lon = swath.longitude.values.ravel()
    lat = swath.latitude.values.ravel()
    ssh = swath.ssh_karin_2.values.ravel()

    resampler = PSFResampler(
        lon_deg=lon, lat_deg=lat, level=level, out_cell_ids=diamond_cell_ids, **kwargs
    )
    resampled = resampler.resample(ssh, lam=lam)
    if verbose:
        print(
            f"{resampled.cg_niters} iterations, "
            f"final residual: {resampled.cg_residual_norms[-1]}"
        )
    return resampled


def reshape_resampled_2d(
    resampled: ResampleResults,
    n_side: int,
    indices: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Wrap resampled data in 2D."""
    if indices is None:
        indices = reindex_ring_to_2d(n_side)

    cell_ids = to_ring(resampled.cell_ids, depth=level)
    i_sort = np.argsort(cell_ids)
    cell_ids = np.take(cell_ids, i_sort)
    ssh_hp = np.take(resampled.cell_data, i_sort)

    ssh_hp_2d = np.zeros((n_side, n_side))
    np.put(ssh_hp_2d, indices, ssh_hp)
    np.put(cell_ids, indices, cell_ids)
    cell_ids = from_ring(cell_ids, depth=level)
    cell_ids = np.reshape(cell_ids, (n_side, n_side))

    return cell_ids, ssh_hp_2d

## Get pixel size

In [ ]:
def get_pixel_size(
    lon: np.ndarray, lat: np.ndarray, deg: bool = False, verbose: bool = True
) -> tuple[float, float]:
    """Get average pixel size."""
    # [(lon, lat), vertical/along-track, horizontal/cross-track]
    r = np.stack([lon, lat], axis=0)

    dr_x = np.diff(r, axis=2)
    dr_y = np.diff(r, axis=1)

    if not deg:
        factor = 6371e3 / 180 * np.pi
        dr_x *= factor
        dr_y *= factor

        # correct radius for longitude
        dr_x[0] *= np.cos(np.deg2rad(r[1, :, 1:]))
        dr_y[0] *= np.cos(np.deg2rad(r[1, 1:, :]))

    dx = np.mean(np.linalg.norm(dr_x, axis=0))
    dy = np.mean(np.linalg.norm(dr_y, axis=0))

    if verbose:
        unit = "°" if deg else "m"
        print(
            f"Resolution horizontal/cross-track: {dx:g}{unit}, "
            f"vertical/along-track: {dy:g}{unit}"
        )

    return dx, dy


def get_pixel_size_swot(
    swath: xr.Dataset,
    cross_track_margin: int = 30,
    deg: bool = False,
    verbose: bool = True,
) -> tuple[float, float]:
    """Get average pixel size over swath. Avoid far end in cross-track."""
    lon = swath.longitude[:, :-cross_track_margin]
    lat = swath.latitude[:, :-cross_track_margin]
    return get_pixel_size(lon, lat, deg=deg, verbose=verbose)

## Get orientation

In [ ]:
def get_track_angle(swath: xr.Dataset) -> float:
    """Get track angle of SWOT swath."""
    center_line = swath.isel(num_pixels=swath.num_pixels.size // 2)
    dx = center_line.longitude.diff("num_lines")
    dy = center_line.latitude.diff("num_lines")
    return float(np.mean(np.atan2(dy, dx)))

def get_up_angle(lon: np.ndarray, lat: np.ndarray) -> float:
    """Get angle of vertical direction for healpix diamond."""
    i_center_col = lon.shape[1] // 2
    center_lon = lon[:, i_center_col]
    center_lat = lat[:, i_center_col]
    dx = np.diff(center_lon)
    dy = np.diff(center_lat)
    return float(np.mean(np.atan2(dy, dx)))

def get_angles_north_aligned(track_angle: float, side: str) -> np.ndarray:
    """Return North, v_sat and look angles for a north aligned plot."""
    look = track_angle + np.pi / 2 if side == "left" else track_angle - np.pi / 2
    return np.asarray([np.pi/2., track_angle, look])

## Select span

In [ ]:
def select_span(
    swath: xr.Dataset, n_side: int, dx: float = 250, dy: float = 250
) -> xr.Dataset:
    """Select span in the swath with the same size as the healpix diamond."""
    nx = int(n_side * cell_size / dx)
    ny = int(n_side * cell_size / dy)
    center_i_line = swath.num_lines.size // 2
    center_i_pixel = swath.num_pixels.size // 2
    iy0 = center_i_line - ny // 2
    iy1 = center_i_line + ny // 2
    ix0 = center_i_pixel - nx // 2
    ix1 = center_i_pixel + ny // 2

    if iy0 < 0 or iy1 > swath.num_lines.size or ix0 < 0 or ix1 > swath.num_pixels.size:
        print("Diamond is too big to fit in current swath :(")

    return swath.isel(
        num_lines=slice(max(0, center_i_line - ny // 2), center_i_line + ny // 2),
        num_pixels=slice(max(0, center_i_pixel - nx // 2), center_i_pixel + nx // 2),
    )

## Compute spectra

In [ ]:
def compute_psd(
    ssh: np.ndarray, dx: float = 250, dy: float = 250
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Compute 2D PSD."""
    ny, nx = ssh.shape
    fft_c = FFTComputer(
        box_size_x=nx, box_size_y=ny, dx=dx, dy=dy, apodizing_kind="hanning"
    )

    psd_2d = fft_c.psd(ssh)
    fx, fy = fft_c.get_frequencies()

    return fx, fy, psd_2d

## WW3

In [ ]:
def get_ww3_spatial(ww3: xr.Dataset) -> xr.DataArray:
    """Compute spatial spectra."""
    efth = ww3.efth.values
    direction = ww3.direction.values
    freq = ww3.frequency.values
    freq1 = ww3.frequency1.values
    freq2 = ww3.frequency2.values
    efth_, direction_ = put_directions_in_order(efth, direction)

    # Transform spectrum units from (f_temp [s], phi [°]) to (f [m], phi [rad])
    # (Important !! output spectrum contains k inside)
    f, phi, ef_phi = transform_from_ftphi_to_fphi(freq, direction_, freq1, freq2, efth_)

    return xr.DataArray(
        np.mean(ef_phi, axis=1),
        dims=["frequency"],
        coords={"frequency": f},
    )

## Study cases

In [ ]:
@dataclass
class StudyCase:
    name: str
    cycle: int
    track: int
    lat: float
    date: dt.date
    side: str = "left"

    def load(self) -> xr.Dataset:
        """Load part of the SWOT swath for this case."""
        return load(cycle=self.cycle, track=self.track, lat=self.lat, side=self.side)

    def load_ww3(self) -> xr.Dataset:
        """Load WW3 data corresponding to case."""
        ww3_file = "/home/chaeck/Data/WaveWatchIII/SWOT_WW3-GLOB-30M_202306_trck.nc"

        ww3 = xr.open_dataset(ww3_file, chunks="auto")
        ww3 = ww3.sel(time=self.date.strftime("%Y%m%d"))

        track_names = ww3.track_name.values
        track_names = np.array(["".join(s) for s in track_names.astype("U")])

        cycle, _, track_names = np.char.partition(track_names, "-").T
        track, _, track_names = np.char.partition(track_names, "-").T
        cycle = cycle.astype("i")
        track = track.astype("i")
        ww3 = ww3.assign_coords(cycle=(["time"], cycle), track=(["time"], track))
        ww3 = ww3.sel(cycle=self.cycle).sel(track=self.track)

        i_time = np.argmin(np.abs(ww3.latitude.values - self.lat))
        ww3 = ww3.isel(time=i_time)

        return ww3


cases_def = [
    StudyCase("long_swells", 546, 19, lat=-46.96, date=dt.date(2023, 6, 9)),
    StudyCase("remote_reflections", 547, 21, lat=-45.4, date=dt.date(2023, 6, 10)),
    StudyCase("crossing_swells", 548, 17, lat=-29.3, date=dt.date(2023, 6, 11)),
]

for case in cases_def:
    case.swath = case.load()
    case.track_angle = get_track_angle(case.swath)
    case.ww3 = case.load_ww3()
    case.ef = get_ww3_spatial(case.ww3)


def get_cases() -> list[StudyCase]:
    cases = [StudyCase(**asdict(case)) for case in cases_def]
    for case, case_def in zip(cases, cases_def, strict=True):
        case.swath = case_def.swath
        case.track_angle = case_def.track_angle
        case.ww3 = case_def.ww3
        case.ef = case_def.ef
    return cases

cases = get_cases()

## All in one

In [ ]:
def all_in_one(
    swath: xr.Dataset,
    lam: float = 0.0,
    sigma_m: float | None = None,
    threshold: float = 0.1,
    verbose: bool = False,
) -> xr.Dataset:
    """Do all processing at once. Return 1D PSD."""
    n_ring = 100
    n_side, diamond_cell_ids = get_diamond(swath, n_ring)

    resampled = resample(
        swath,
        diamond_cell_ids,
        lam=lam,
        threshold=threshold,
        sigma_m=sigma_m,
    )

    cell_ids_2d, ssh_hp_2d = reshape_resampled_2d(resampled, n_side)

    diamond_lon_2d, diamond_lat_2d = healpix_geo.ring.healpix_to_lonlat(
        cell_ids_2d, depth=level, ellipsoid="WGS84"
    )

    dx, dy = get_pixel_size_swot(swath, verbose=verbose)
    dx_hp, dy_hp = get_pixel_size(diamond_lon_2d, diamond_lat_2d)

    span = select_span(swath, n_side, dx, dy)

    fx, fy, psd_2d = compute_psd(span.ssh_karin_2.values, dx, dy)
    fr, psd_1d = average_psd_azimuth(psd_2d, fx, fy)

    fx_hp, fy_hp, psd_2d_hp = compute_psd(ssh_hp_2d, dx_hp, dy_hp)
    fr_hp, psd_1d_hp = average_psd_azimuth(psd_2d_hp, fx_hp, fy_hp)

    ds = xr.Dataset(
        {
            "psd": (("fr"), psd_1d),
            "psd_hp": (("fr_hp"), psd_1d_hp),
        },
        coords={
            "fr": fr,
            "fr_hp": fr_hp,
        },
    )
    return ds

# Single image

First case study (long-swells).

## Load data

In [ ]:
case = cases_def[0]
swath = case.swath

fig, ax = plt.subplots(figsize=(4, 3), layout="constrained", dpi=200)
swath.ssh_karin_2.plot.pcolormesh(
    x="longitude", y="latitude", center=False, cmap="Blues_r"
)

add_look_arrows(
    ax,
    *get_angles_north_aligned(case.track_angle, case.side),
    label_offsets={"look": (8, 4)},
    box_kwargs={"pad": 0.8},
)

## Select diamond

Find a large healpix diamond centered on the swath in and find reshape it in 2D

In [ ]:
n_ring = 100
n_side, diamond_center_cell_id, diamond_cell_ids = get_diamond(swath, n_ring)

fig, ax = plt.subplots(figsize=(4, 3), dpi=200)
swath.ssh_karin_2.plot.pcolormesh(
    x="longitude",
    y="latitude",
    center=False,
    cmap="Blues_r",
    add_labels=False,
)
vertices = np.stack(
    healpix_geo.nested.vertices(diamond_cell_ids, depth=level, ellipsoid="WGS84"),
    axis=-1,
)
ax.add_artist(PolyCollection(vertices, ec="r", lw=0.2, fc="none", alpha=0.5))

add_look_arrows(
    ax,
    *get_angles_north_aligned(case.track_angle, case.side),
    label_offsets={"look": (8, 4)},
    box_kwargs={"pad": 0.8},
)

ax.set_aspect("equal")


## Find SWOT pixel size

We need it to select a region of the same size as our healpix diamond.
We also need it to compute frequencies accurately.


In [ ]:
cross_track_margin = 30
# Avoid that many pixels on the close edge cross-track when averaging pixel size

# [(lon, lat), along-track, cross-track]
r = np.stack([swath.longitude, swath.latitude], axis=0)

factor = 6371e3 / 180 * np.pi
dr_x = np.diff(r, axis=2) * factor
dr_y = np.diff(r, axis=1) * factor

# correct radius for longitude
dr_x[0] *= np.cos(np.deg2rad(r[1, :, 1:]))
dr_y[0] *= np.cos(np.deg2rad(r[1, 1:, :]))

dr_x = np.linalg.norm(dr_x, axis=0)
dr_y = np.linalg.norm(dr_y, axis=0)

In [ ]:
fig, axes = plt.subplots(
    1, 2, sharex=True, sharey=True, layout="constrained", figsize=(3.7, 3.7), dpi=200
)
labels = (["Δ(Cross-track)", "width"], ["Δ(Along-track)", "height"])

for ax, dr, (title, unit) in zip(axes, [dr_x, dr_y], labels):
    vmin = np.quantile(dr, 0.01)
    vmax = np.quantile(dr, 0.99)
    im = ax.imshow(dr, origin="lower", vmin=vmin, vmax=vmax, cmap=tc.rainbow_PuRd)

    divider = make_axes_locatable(ax)
    ax_avg_x = divider.append_axes("top", 0.42, pad=0.05, sharex=ax)
    ax_avg_y = divider.append_axes("right", 0.35, pad=0.05, sharey=ax)
    ax_avg_x.tick_params(labelbottom=False, labelleft=False, left=False, bottom=False)
    ax_avg_y.tick_params(labelbottom=False, labelleft=False, left=False, bottom=False)

    ax_avg_x.plot(np.mean(dr, axis=0), lw=0.7)
    ax_avg_y.plot(np.mean(dr, axis=1), np.arange(dr.shape[0]), lw=0.7)

    ax_avg_x.set_title(title)

    for ax_ in (ax, ax_avg_x):
        ax_.axvline(
            dr.shape[1] - cross_track_margin, lw=0.7, alpha=0.7, color=tc.bright.red
        )

    fig.colorbar(im, ax=ax, location="bottom", label=f"Pixel {unit}")
axes[0].set_ylabel("Along-track")

In [ ]:
fig, axes = plt.subplots(
    2, 1, sharex=True, figsize=(4, 2.3), dpi=200, layout="constrained"
)
fig.suptitle("Along-track average")

for ax, dr, label in zip(
    axes, [dr_x, dr_y], ("width\n(cross-track)", "height\n(along-track)")
):
    ax.set_ylabel("pixel " + label)
    ax.plot(np.mean(dr, axis=0))
    ax.axvline(
        dr.shape[1] - cross_track_margin, color=tc.bright.red, alpha=0.7
    )
    ax.axhline(np.mean(dr[:, :-cross_track_margin]), color="k")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f"{x:.1f}"))

axes[1].set_xlabel("Cross-track pixel index")

In [ ]:
dx, dy = get_pixel_size_swot(swath, cross_track_margin=cross_track_margin)
dx_deg, dy_deg = get_pixel_size_swot(
    swath, cross_track_margin=cross_track_margin, deg=True
)

## Select span

with the same size (in meters) as the healpix diamond.

In [ ]:
span = select_span(swath, n_side, dx, dy)

## Resample

and reshape to 2D

In [ ]:
resampled = resample(swath, diamond_cell_ids, sigma_m=275, lam=0.2)
cell_ids = resampled.cell_ids
ssh_hp = resampled.cell_data
cell_ids_2d, ssh_hp_2d = reshape_resampled_2d(resampled, n_side)

diamond_lon_2d, diamond_lat_2d = healpix_geo.nested.healpix_to_lonlat(
    cell_ids_2d, depth=level, ellipsoid="WGS84"
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(6, 3),
    layout="constrained",
    sharex=True,
    sharey=True,
    dpi=150,
)
ax1, ax2 = axes

vmin = float(swath.ssh_karin_2.min())
vmax = float(swath.ssh_karin_2.max())

im = span.ssh_karin_2.plot.pcolormesh(
    x="longitude",
    y="latitude",
    vmin=vmin,
    vmax=vmax,
    ax=ax1,
    add_labels=False,
    add_colorbar=False,
    cmap="Blues_r",
)
ax1.set_title("Input span")

ax2.pcolormesh(
    diamond_lon_2d, diamond_lat_2d, ssh_hp_2d, vmin=vmin, vmax=vmax, cmap="Blues_r"
)
ax2.set_title("Healpix (2D)")

# ax1.set_xlim(np.min(diamond_lon), np.max(diamond_lon))
# ax1.set_ylim(np.min(diamond_lat), np.max(diamond_lat))

up_angle = get_up_angle(diamond_lon_2d, diamond_lat_2d)
add_look_arrow(
    ax2,
    up_angle,
    "up",
    box_size=0.2,
    label_kwargs={"ha": "left"},
    label_offset=(2, 2),
)

fig.colorbar(im, ax=axes, label="[m]")

for ax in axes:
    ax.set_aspect("equal")
    ax.grid(True)

## Check Healpix pixel size

In [ ]:
# [(lon, lat), vertical, horizontal]
r = np.stack([diamond_lon_2d, diamond_lat_2d], axis=0)

factor = 6371e3 / 180 * np.pi
dr_x = np.diff(r, axis=2) * factor
dr_y = np.diff(r, axis=1) * factor

# correct radius for longitude
dr_x[0] *= np.cos(np.deg2rad(r[1, :, 1:]))
dr_y[0] *= np.cos(np.deg2rad(r[1, 1:, :]))

dr_x = np.linalg.norm(dr_x, axis=0)
dr_y = np.linalg.norm(dr_y, axis=0)

In [ ]:
fig, axes = plt.subplots(
    1, 2, sharex=True, sharey=True, layout="constrained", figsize=(3.7, 2.5), dpi=200
)
labels = (["Δ(Horizontal)", "width"], ["Δ(Vertical)", "height"])

for ax, dr, (title, unit) in zip(axes, [dr_x, dr_y], labels):
    vmin = np.quantile(dr, 0.01)
    vmax = np.quantile(dr, 0.99)
    im = ax.imshow(dr, origin="lower", vmin=vmin, vmax=vmax, cmap=tc.rainbow_PuRd)

    divider = make_axes_locatable(ax)
    ax_avg_x = divider.append_axes("top", 0.42, pad=0.05, sharex=ax)
    ax_avg_y = divider.append_axes("right", 0.35, pad=0.05, sharey=ax)
    ax_avg_x.tick_params(labelbottom=False, labelleft=False, left=False, bottom=False)
    ax_avg_y.tick_params(labelbottom=False, labelleft=False, left=False, bottom=False)

    ax_avg_x.plot(np.mean(dr, axis=0), lw=0.7)
    ax_avg_y.plot(np.mean(dr, axis=1), np.arange(dr.shape[0]), lw=0.7)

    ax_avg_x.set_title(title)

    fig.colorbar(im, ax=ax, location="bottom", label=f"Pixel {unit}")
axes[0].set_ylabel("Vertical")

In [ ]:
dx_hp, dy_hp = get_pixel_size(diamond_lon_2d, diamond_lat_2d)
dx_hp_deg, dy_hp_deg = get_pixel_size(diamond_lon_2d, diamond_lat_2d, deg=True)

## Spectra

In [ ]:
# Input
fx, fy, psd_2d = compute_psd(span.ssh_karin_2.values, dx, dy)
fx *= 1000
fy *= 1000
fr, psd_1d = average_psd_azimuth(psd_2d, fx, fy)

# Resampled
fx_hp, fy_hp, psd_2d_hp = compute_psd(ssh_hp_2d, dx_hp, dy_hp)
fx_hp *= 1000
fy_hp *= 1000
fr_hp, psd_1d_hp = average_psd_azimuth(psd_2d_hp, fx_hp, fy_hp)

In [ ]:
fig, axes = plt.subplots(
    1, 2, layout="constrained", figsize=(6, 3.5), dpi=200, sharex=True, sharey=True
)
ax1, ax2 = axes

ax1.pcolormesh(fx, fy, psd_2d, norm="log")
ax1.set_title("SSH spectrum (SWOT)")
ax2.pcolormesh(fx_hp, fy_hp, psd_2d_hp, norm="log")
ax2.set_title("SSH spectrum (Healpix)")

# ax1.set_xlim(fx[0], fx[-1])
# ax1.set_ylim(fy[0], fy[-1])

for ax in axes:
    ax.set_aspect("equal")
    ax.grid(False)

    for dist in [1000]:
        ax.add_artist(
            plt.Circle(
                (0, 0),
                radius=1 / (dist / 1000),
                lw=0.7,
                ec="w",
                alpha=0.4,
                fc="none",
            )
        )

ax1.set_ylabel("[cycle/km]")

add_look_arrows(
    ax1,
    *get_angles_north_aligned(case.track_angle, case.side)
    + np.pi / 2
    - case.track_angle,
    box_size=0.15,
    box_kwargs={"pad": 2},
    label_offsets=dict(look=(2, 2)),
    label_kwargs=dict(vsat=dict(ha="left")),
)

up_angle = get_up_angle(diamond_lon_2d, diamond_lat_2d)
add_look_arrows(
    ax2,
    *get_angles_north_aligned(case.track_angle, case.side)
    - np.pi / 2
    - case.track_angle
    + up_angle,
    box_size=0.15,
    box_kwargs={"pad": 2},
    label_offsets=dict(look=(2, 2)),
    label_kwargs=dict(vsat=dict(ha="left")),
)

In [ ]:
fig, ax = plt.subplots(layout="constrained", figsize=(3, 2), dpi=200)
ax.plot(fr, psd_1d, label="SWOT")
ax.plot(fr_hp, psd_1d_hp, label="Healpix")
ax.plot(case.ef.frequency*1000, case.ef*15, color="k", label="WWIII") 
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(fr_hp[0], 16)
# ax.set_ylim(1e-6, None)

## Shift of peak

In [ ]:
from scipy.optimize import curve_fit


def get_gaussian_peak(
    x: np.ndarray, y: np.ndarray, guess: float, bounds: tuple[float, float]
) -> float:
    i_bounds = np.searchsorted(x, bounds)

    x = x[slice(*i_bounds)]
    y = y[slice(*i_bounds)]
    x = np.log(x)
    y = np.log(y)

    def gauss(x, B, A, x0, std):
        return B + A * np.exp(-0.5 * (x - x0) ** 2 / std**2)

    ymin = np.min(y)
    ymax = np.max(y)
    p0 = [ymin, ymax - ymin, np.log(guess), 0.1]

    bounds = [
        (-np.inf, 1e-3, np.log(bounds[0]), 1e-3),
        (np.inf, 20, np.log(bounds[1]), 10),
    ]
    p, _ = curve_fit(gauss, x, y, p0, bounds=bounds)
    return np.exp(p[2])


guess = 1.5
bounds = [0.8, 2]
peak = get_gaussian_peak(fr, psd_1d, guess=guess, bounds=bounds)
peak_hp = get_gaussian_peak(fr_hp, psd_1d_hp, guess=guess, bounds=bounds)
# peak_nufft = get_gaussian_peak(fr_nufft*1000, psd_1d_nufft, guess=guess, bounds=bounds)

print(f"Peak SWOT: {peak:.3f} km-1")
print(f"Peak resampled: {peak_hp:.3f} km-1")
print(f"Shift de {abs(1.0 / peak - 1.0 / peak_hp) * 1000:.2f} m")

fig, ax = plt.subplots(layout="constrained", figsize=(3, 2), dpi=200)
ax.plot(fr, psd_1d, color=tc.bright.blue, label="SWOT")
ax.plot(fr_hp, psd_1d_hp, color=tc.bright.red, label="Healpix")
# ax.plot(fr_nufft*1000, 5e5*psd_1d_nufft, color=tc.bright.green, label="NUFFT")

ax.axvline(peak, color=tc.bright.blue, lw=0.7)
ax.axvline(peak_hp, color=tc.bright.red, lw=0.7)
# ax.axvline(peak_nufft, color=tc.bright.green, lw=0.7)


ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")

## Test computing spectra with healpix_analyse

In [ ]:
from healpix_analyse.fft_local import LocalFFT
from scipy.signal.windows import hann, tukey

transform = LocalFFT(
    cell_ids,
    level=level,
    ellipsoid="WGS84",
    device="cpu",
)

# win = hann(n_side)
# window = win[:, None] * win

data = ssh_hp
data = data - np.mean(data)
# data = data * window

grid = transform.project(data.ravel())
mask = transform.coverage_mask.numpy()

weighted = grid * mask
spectrum = np.fft.fft2(weighted, norm="ortho")

# window_energy = np.mean(window[mask]**2)
# spectrum = spectrum / np.sqrt(window_energy)
power = np.abs(spectrum)**2

# To radial
frequency, radial_power = transform.ps(spectrum)
frequency = frequency / 6371e3

In [ ]:
fig, axes = plt.subplots(1, 2, layout="constrained", figsize=(4, 2), dpi=200)
ax1, ax2 = axes

vmin = np.min(weighted)
vmax = np.max(weighted)
vmax_abs = max(abs(vmin), abs(vmax))

ax1.imshow(weighted, vmin=-vmax_abs, vmax=vmax_abs, cmap="Blues_r")
ax2.imshow(power, norm="log")

for ax in axes:
    ax.set_aspect("equal")

In [ ]:
def plot(ax, fr, psd, factor=1, **kwargs):
    norm = np.sum(fr * psd)
    ax.plot(fr, psd/norm*factor, **kwargs)

fig, ax = plt.subplots(layout="constrained", figsize=(3, 2), dpi=200)
lw = 1

plot(ax, fr, psd_1d, lw=lw, label="SWOT")
plot(ax, fr_hp, psd_1d_hp, lw=lw, label="Naive")
# plot(ax, case.ef.frequency*1000, case.ef, color="k", lw=lw, label="WWIII")
plot(ax, frequency*1000, radial_power, factor=8, lw=lw, label="hp_analyse")
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(fr_hp[0], fr_hp[-1])
ax.set_ylim(1e-6, None)
ax.set_xlabel("[cycle/km]")

## Spectra with pyNFFT3

In [ ]:
dist = healpix_geo.nested.angular_distances(
    from_=np.full([cell_ids_2d.size], diamond_center_cell_id),
    to_=cell_ids_2d.ravel(),
    depth=level,
)
dist = np.fabs(dist * 6371e3)
dist = dist.reshape((n_side, n_side))

lon_c, lat_c = healpix_geo.nested.healpix_to_lonlat(
    diamond_center_cell_id, level, ellipsoid="WGS84"
)
x_deg = diamond_lon_2d - lon_c
y_deg = diamond_lat_2d - lat_c
# cells_lon, cells_lat = healpix_geo.nested.healpix_to_lonlat(cell_ids, level, ellipsoid="WGS84")
# x_deg = cells_lon - lon_c
# y_deg = cells_lat - lat_c
x_m = x_deg * 6371e3 * np.pi / 180 * np.cos(np.deg2rad(diamond_lat_2d))
y_m = y_deg * 6371e3 * np.pi / 180
dist_ = np.sqrt(x_m**2 + y_m**2)

vmin = min(np.min(dist), np.min(dist_))
vmax = max(np.max(dist), np.max(dist_))

fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.imshow(dist, vmin=vmin, vmax=vmax)
ax2.imshow(dist_, vmin=vmin, vmax=vmax)

# healpix_grid = {"indexing_scheme": "nested", "level": level, "ellipsoid": "WGS84"}
# hp.pcolormesh(
#     cell_ids_2d.ravel(),
#     dist.ravel(),
#     healpix_grid=healpix_grid,
#     level_tile=7,
#     ax=ax1,
# )
# hp.pcolormesh(
#     cell_ids,
#     dist_,
#     healpix_grid=healpix_grid,
#     level_tile=7,
#     ax=ax2,
# )

In [ ]:
from pyNFFT3 import NFFT

# f = span.ssh_karin_2.values
f = ssh_hp_2d
Nfreq = np.asarray([198, 200])

# cut = np.concat([np.asarray([146, 6890, 6890, 6891, 6892]), np.arange(20200, 20300)])
# cell_ids_cut = np.delete(cell_ids, cut)
# f = np.delete(f, cut)

# xy = np.stack([diamond_lon_2d.ravel(), diamond_lat_2d.ravel()], axis=-1)
# x = diamond_lon_2d
# y = diamond_lat_2d
x = x_m
y = y_m
xmin = np.min(x)
xmax = np.max(x)
ymin = np.min(y)
ymax = np.max(y)

x_norm = (x - (xmin+xmax)/2) / (xmax-xmin)
y_norm = (y - (ymin+ymax)/2) / (ymax-ymin)
xy_norm = np.stack([x_norm, y_norm], axis=-1)

win_x = hann(f.shape[1], sym=True)
win_y = hann(y.shape[0], sym=True)
window = win_y[:, None] * win_x

f = f - np.mean(f)
# f = f + 0.05*np.sin(2*np.pi / 0.1 * xx)
f = f * window

In [ ]:
plan = NFFT(N=Nfreq, M=f.size)

plan.x = xy_norm

plan.f = np.ascontiguousarray(f.astype("complex128").ravel())

plan.nfft_adjoint()

psd_nufft = (np.abs(plan.fhat)**2).reshape(Nfreq).T

fx_nufft = np.arange(-Nfreq[0] // 2, Nfreq[0] // 2) / (xmax - xmin)
fy_nufft = np.arange(-Nfreq[1] // 2, Nfreq[1] // 2) / (ymax - ymin)
fx_nufft *= 1000
fy_nufft *= 1000

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(4, 1.8), layout="constrained", dpi=250)
ax1, ax2 = axes

vmin = np.nanmin(np.real(f))
vmax = np.nanmax(np.real(f))
vabs_max = max(abs(vmin), abs(vmax))

im = ax1.pcolormesh(
    x_m / 1000,
    y_m / 1000,
    np.real(f),
    cmap="Blues_r",
)
fig.colorbar(im, ax=ax1)

# hp.pcolormesh(
#     cell_ids,
#     f,
#     healpix_grid=healpix_grid,
#     ax=ax1
# )

im = ax2.pcolormesh(fx_nufft, fy_nufft, psd_nufft, norm="log")
fig.colorbar(im, ax=ax2)


for dist in [1000]:
    ax2.add_artist(
        plt.Circle(
            (0, 0),
            radius=1 / (dist / 1000),
            lw=0.7,
            ec="w",
            alpha=0.4,
            fc="none",
        )
    )


ax1.set_xlabel("[km]")
ax2.set_xlabel("[cycle/km]")

for ax in axes:
    ax.set_aspect("equal")

In [ ]:
fr_nufft, psd_1d_nufft = average_psd_azimuth(psd_nufft, fx_nufft, fy_nufft)

In [ ]:
def plot(ax, fr, psd, factor=1, **kwargs):
    norm = np.sum(fr * psd)
    ax.plot(fr, psd/norm*factor, **kwargs)

fig, ax = plt.subplots(layout="constrained", figsize=(4, 2), dpi=200)
lw = 1.

plot(ax, fr, psd_1d, lw=lw, label="SWOT")
plot(ax, fr_hp, psd_1d_hp, lw=lw, label="Naive")
# plot(ax, case.ef.frequency * 1000, case.ef, color="k", lw=lw, label="WWIII")
plot(ax, frequency*1000, radial_power, factor=3, lw=lw, label="Gnomonic")
plot(ax, fr_nufft, psd_1d_nufft, lw=lw, label="NUFFT")
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("[cycle/km]")
# ax.set_xlim(fr_hp[0], 16)
# ax.set_ylim(1e-6, None)

# Multiple chunks along swath

Compute spectra for multiple chunk along the swath, average spectras.
The swath must not pass through wrapping longitude (180°E), otherwise the code reshaping the chunk in 2d will fail silently.

In [ ]:
track = SWOT(cycle=546, track=19, version="PGC0")
swot_file = track.get_file()
swath = xr.open_dataset(swot_file, group="left", chunks=-1)

swath = swath.isel(num_lines=slice(42_000, 62_000))

# Left swath, we flip to have ascending longitude
swath = swath.isel(num_pixels=slice(None, None, -1))
if track.pass_type == "descending":
    swath = swath.isel(num_lines=slice(None, None, -1))

swath.longitude.load()
swath.latitude.load()
swath.ssh_karin_2.load()
swath.ancillary_surface_classification_flag.load()

swath = swath.dropna("num_lines", how="all", subset=["longitude"])
swath = swath.dropna("num_pixels", how="any", subset=["longitude"])

# swath = swath.isel(num_lines=slice(0, swath.num_pixels.size))

swath["ssh_karin_2"] -= swath.ssh_karin_2.mean()

swath

In [ ]:
center_line = swath.isel(num_pixels=swath.num_pixels.size // 2)
fig, ax = plt.subplots(
    layout="constrained",
    figsize=(5, 3),
    dpi=200,
    subplot_kw={"projection": ccrs.PlateCarree()},
)
ax.plot(center_line.longitude, center_line.latitude)
#swath.ssh_karin_2.plot.pcolormesh(x="longitude", y="latitude")
ax.set_global()
ax.coastlines()

In [ ]:
def plot_check_reshape(i_chunk, span, ssh_hp, ssh_hp_2d, chunk_lon_2d, chunk_lat_2d):
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(8.5, 3),
        layout="constrained",
        sharex=True,
        sharey=True,
        dpi=150,
    )
    ax1, ax2, ax3 = axes

    vmin = float(span.ssh_karin_2.min())
    vmax = float(span.ssh_karin_2.max())

    im = span.ssh_karin_2.plot.pcolormesh(
        x="longitude",
        y="latitude",
        vmin=vmin,
        vmax=vmax,
        ax=ax1,
        add_labels=False,
        add_colorbar=False,
    )
    ax1.set_title("SWOT")

    vertices = np.stack(
        healpix_geo.nested.vertices(ssh_hp.cell_ids, depth=level, ellipsoid="WGS84"),
        axis=-1,
    )
    colorizer = plt.Colorizer(norm=plt.Normalize(vmin=vmin, vmax=vmax))
    colors = colorizer.to_rgba(ssh_hp.cell_data)
    polys = PolyCollection(vertices, ec=colors, fc=colors)
    ax2.add_collection(polys)
    ax2.set_title("Healpix")

    # ax1.add_collection(PolyCollection(vertices, ec="w", fc="none", lw=0.5, alpha=0.2))

    ax3.pcolormesh(chunk_lon_2d, chunk_lat_2d, ssh_hp_2d, vmin=vmin, vmax=vmax)
    ax3.set_title("Healpix (2D)")

    ax1.set_xlim(np.min(diamond_lon), np.max(diamond_lon))
    ax1.set_ylim(np.min(diamond_lat), np.max(diamond_lat))

    fig.colorbar(im, ax=axes, label="[m]")

    for ax in axes:
        ax.set_aspect("equal")
        ax.grid(True)

    fig.savefig(path.join(plot_dir, f"chunk_{i_chunk}_check_reshape.png"))
    plt.close(fig)

In [ ]:
def plot_psd_2d(fx, fy, psd_2d, fx_hp, fy_hp, psd_2d_hp):
    fig, axes = plt.subplots(
        1, 2, layout="constrained", figsize=(5, 2.5), dpi=150, sharex=True, sharey=True
    )
    ax1, ax2 = axes

    ax1.pcolormesh(fx, fy, psd_2d, norm="log")
    ax1.set_title("SSH spectrum (SWOT)")
    ax2.pcolormesh(fx_hp, fy_hp, psd_2d_hp, norm="log")
    ax2.set_title("SSH spectrum (Healpix)")

    ax1.set_xlim(fx[0], fx[-1])
    ax1.set_ylim(fy[0], fy[-1])

    for ax in axes:
        ax.set_aspect("equal")
        ax.grid(False)

        for dist in [1000]:
            ax.add_artist(
                plt.Circle(
                    (0, 0),
                    radius=1 / (dist / 1000),
                    lw=0.7,
                    ec="w",
                    alpha=0.4,
                    fc="none",
                )
            )

    ax1.set_ylabel("[cycle/km]")

    fig.savefig(path.join(plot_dir, f"chunk_{i_chunk}_fft_2d.png"))
    plt.close(fig)

In [ ]:
def plot_psd_1d(fr, psd_1d, fr_hp, psd_1d_hp):
    fig, ax = plt.subplots(layout="constrained", figsize=(3, 2), dpi=200)
    ax.plot(fr, psd_1d, label="SWOT")
    ax.plot(fr_hp, psd_1d_hp, label="Healpix")
    ax.legend()
    ax.set_xscale("log")
    ax.set_yscale("log")

    fig.savefig(path.join(plot_dir, f"chunk_{i_chunk}_fft_1d.png"))
    plt.close(fig)

In [ ]:
n_ring = 90
n_side = 2 * n_ring + 1
ring2d_indices = reindex_ring_to_2d(n_side)

threshold = 0.1
lam = 0.05
sigma_m = cell_size / 2

chunk_step = 2000
chunk_indices = np.arange(0, swath.num_lines.size + chunk_step, chunk_step)
chunk_indices = ((chunk_indices[1:] + chunk_indices[:-1]) / 2).astype("i")

do_plots = False

psd_list = []
psd_hp_list = []

for i_chunk, chunk_i0 in enumerate(chunk_indices):
    print(f"Chunk #{i_chunk + 1}/{len(chunk_indices)}")
    _, diamond_cell_ids = get_diamond(swath, n_ring, center_i_line=chunk_i0)

    if diamond_cell_ids.size != n_side**2:
        print("Chunk has incorrect size, skipping")
        continue

    swath_chunk = swath.isel(num_lines=slice(chunk_i0 - 150, chunk_i0 + 150))

    resampled = resample(
        swath_chunk, diamond_cell_ids, lam=lam, threshold=threshold, sigma_m=sigma_m
    )

    if np.any(~np.isfinite(resampled.cell_data)):
        print("Found Nan in resampled data, skipping")
        continue

    cell_ids, ssh_hp_2d = reshape_resampled_2d(resampled, n_side, ring2d_indices)

    diamond_lon, diamond_lat = healpix_geo.ring.healpix_to_lonlat(
        cell_ids, depth=level, ellipsoid="WGS84"
    )
    diamond_lon_2d = np.reshape(diamond_lon, (n_side, n_side))
    diamond_lat_2d = np.reshape(diamond_lat, (n_side, n_side))

    dx, dy = get_resolution(swath)
    span = select_span(swath, n_side, dx, dy)

    if do_plots:
        plot_check_reshape(
            i_chunk, span, resampled, ssh_hp_2d, diamond_lon_2d, diamond_lat_2d
        )

    fx, fy, psd_2d = compute_psd(span.ssh_karin_2.values, dx, dy)
    fr, psd_1d = average_psd_azimuth(psd_2d, fx, fy)

    fx_hp, fy_hp, psd_2d_hp = compute_psd(ssh_hp_2d, cell_size, cell_size)
    fr_hp, psd_1d_hp = average_psd_azimuth(psd_2d_hp, fx_hp, fy_hp)

    if do_plots:
        plot_psd_1d(fr, psd_1d, fr_hp, psd_1d_hp)
        plot_psd_2d(fx, fy, psd_2d, fx_hp, fy_hp, psd_2d_hp)

    psd_list.append(psd_1d)
    psd_hp_list.append(psd_1d_hp)

In [ ]:
psd_1d_avg = np.mean(np.stack(psd_list), axis=0)
psd_1d_hp_avg = np.mean(np.stack(psd_hp_list), axis=0)

In [ ]:
fig, ax = plt.subplots(layout="constrained", figsize=(3, 2), dpi=200)
ax.plot(fr, psd_1d_avg, label="SWOT")
ax.plot(fr_hp, psd_1d_hp_avg, label="Healpix")
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
fig.savefig(
    path.join(plot_dir, f"spectra_comp_thr{threshold:.2f}_lam{lam:.2f}_sigdef.png")
)

# Study cases

Use 3 different study cases (selected from Ardhuin 2024).

In [ ]:
cases = get_cases()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 2.8), dpi=250, layout="constrained")
fig.suptitle("Input data")

for ax, c in zip(axes.flat, cases):
    c.swath.ssh_karin_2.plot.pcolormesh(
        x="longitude",
        y="latitude",
        add_labels=False,
        center=False,
        cmap="Blues_r",
        ax=ax,
    )
    ax.set_title(f"{c.name}\n{c.cycle}:{c.track}, {format_latlon(c.lat, 'lat')}")
    ax.set_aspect("equal")

    add_look_arrows(
        ax,
        *get_angles_north_aligned(c.track_angle, c.side),
        arrows_kwargs={"width": 1.5},
        label_kwargs={"north": {"ha": "center"}, "vsat": {"va": "center"}},
        label_offsets={"look": (8, 4)},
        box_kwargs={"pad": 0.8},
    )

In [ ]:
# Resample, reshape, select span
n_ring = 100
n_side = 2 * n_ring + 1
indices = reindex_ring_to_2d(n_side)

for case in cases:
    case.n_side, case.diamond_cell_ids = get_diamond(case.swath, n_ring)

    case.resampled = resample(
        case.swath, case.diamond_cell_ids, lam=0.5, threshold=0.1, sigma_m=None
    )

    case.cell_ids, case.ssh_hp_2d = reshape_resampled_2d(
        case.resampled, n_side, indices
    )

    diamond_lon, diamond_lat = healpix_geo.ring.healpix_to_lonlat(
        case.cell_ids, depth=level, ellipsoid="WGS84"
    )
    case.diamond_lon_2d = np.reshape(diamond_lon, (n_side, n_side))
    case.diamond_lat_2d = np.reshape(diamond_lat, (n_side, n_side))

    case.dx, case.dy = get_pixel_size_swot(case.swath)
    case.dx_hp, case.dy_hp = get_pixel_size(case.diamond_lon_2d, case.diamond_lat_2d)
    case.span = select_span(case.swath, n_side, case.dx, case.dy)

In [ ]:
fig, axes = plt.subplots(
    2, 3, figsize=(6, 4), dpi=250, layout="constrained", sharex="col", sharey="col"
)
fig.suptitle("Selection for spectra")

for axes_col, c in zip(axes.T, cases):
    ax1, ax2 = axes_col
    vmin = float(c.span.ssh_karin_2.min())
    vmax = float(c.span.ssh_karin_2.max())
    kwargs = dict(cmap="Blues_r", vmin=vmin, vmax=vmax)

    im = ax1.pcolormesh(c.span.longitude, c.span.latitude, c.span.ssh_karin_2, **kwargs)
    ax2.pcolormesh(c.diamond_lon_2d, c.diamond_lat_2d, c.ssh_hp_2d, **kwargs)
    fig.colorbar(im, ax=axes_col, location="bottom")
    ax1.set_title(c.name)

    up_angle = get_up_angle(c.diamond_lon_2d, c.diamond_lat_2d)
    add_look_arrow(
        ax2,
        up_angle,
        "up",
        box_size=0.12,
        box_kwargs={"pad": 0.05},
        label_kwargs={"ha": "left"},
        label_offset=(2, 2),
    )

    for ax in axes_col:
        ax.set_aspect("equal")
    
for ax, label in zip(axes[:, -1], ["Input (cut)", "Resampled"]):
    ax.annotate(
        label,
        xy=(1, 0.5),
        xycoords="axes fraction",
        xytext=(3, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        rotation=90,
        weight="bold",
    )

In [ ]:
# Compute PSD
for case in cases:
    case.fx, case.fy, case.psd_2d = compute_psd(
        case.span.ssh_karin_2.values, case.dx, case.dy
    )
    case.fr, case.psd_1d = average_psd_azimuth(case.psd_2d, case.fx, case.fy)

    case.fx_hp, case.fy_hp, case.psd_2d_hp = compute_psd(case.ssh_hp_2d, 198, 198)
    case.fr_hp, case.psd_1d_hp = average_psd_azimuth(
        case.psd_2d_hp, case.fx_hp, case.fy_hp
    )

In [ ]:
fig, axes = plt.subplots(
    2, 3, figsize=(6, 5), dpi=250, layout="constrained", sharex="col", sharey="col"
)
fig.suptitle("2D PSD")

for axes_col, c in zip(axes.T, cases):
    ax1, ax2 = axes_col

    vmin = min(c.psd_2d.min(), c.psd_2d_hp.min())
    vmax = max(c.psd_2d.max(), c.psd_2d_hp.max())

    im = ax1.pcolormesh(c.fx, c.fy, c.psd_2d, norm="log", vmin=vmin, vmax=vmax)
    ax2.pcolormesh(c.fx_hp, c.fy_hp, c.psd_2d_hp, norm="log", vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=axes_col, location="bottom")
    ax1.set_title(c.name)

    add_look_arrows(
        ax1,
        *get_angles_north_aligned(c.track_angle, c.side) + np.pi / 2 - c.track_angle,
        box_size=0.12,
        box_kwargs={"pad": 1},
        label_offsets=dict(look=(8, 4)),
        label_kwargs=dict(vsat=dict(ha="left")),
    )

    up_angle = get_up_angle(c.diamond_lon_2d, c.diamond_lat_2d)
    add_look_arrows(
        ax2,
        *get_angles_north_aligned(c.track_angle, c.side)
        - np.pi / 2
        - c.track_angle
        + up_angle,
        box_size=0.12,
        box_kwargs={"pad": 1},
        label_offsets=dict(look=(2, 2)),
        label_kwargs=dict(vsat=dict(ha="left", va="center"), north=dict(ha="center")),
    )

    for ax in axes_col:
        ax.set_aspect("equal")

for ax, label in zip(axes[:, -1], ["PSD (Input)", "PSD (Resampled)"]):
    ax.annotate(
        label,
        xy=(1, 0.5),
        xycoords="axes fraction",
        xytext=(3, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        rotation=90,
        weight="bold",
    )

for ax in axes.flat:
    ax.add_artist(
        plt.Circle(
            (0, 0),
            radius=1.0,
            lw=0.7,
            ec="w",
            alpha=0.4,
            fc="none",
        )
    )

In [ ]:
fig, axes = plt.subplots(
    3, 1, figsize=(4, 4), dpi=300, layout="constrained", sharex=True
)
fig.suptitle("1D PSD")

cset = tc.bright

for ax, c in zip(axes, cases):
    ax.yaxis.set_label_position("right")
    ax.set_ylabel(c.name)
    ax.plot(c.fr, c.psd_1d, color=cset.blue, label="Input")
    ax.plot(c.fr_hp, c.psd_1d_hp, color=cset.red, label="Resampled")
    ax.plot(c.ef.frequency*1000, c.ef/1000, color="k", lw=0.7, label="WW3")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(c.fr[0], 80)
    ax.set_ylim(np.min(c.psd_1d), c.psd_1d[0])
axes[0].legend()
axes[-1].set_xlabel("[$km^{-1}$]")

# Varying parameters

## Sigma

In [ ]:
swath = cases[0].load()

lam = 0.5
psd_list = []
for sigma in [130, 175, 198, 250, 300, 400]:
    psd = all_in_one(swath, lam=lam, sigma_m=sigma)
    psd["psd_hp"] = psd.psd_hp.expand_dims(sigma=[sigma])
    psd_list.append(psd)
psd = xr.concat(psd_list, "sigma", data_vars=None)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 2.5), dpi=200)
fig.suptitle(rf"Varying σ ($\lambda = {lam})$")

ax.set_xscale("log")
ax.set_yscale("log")
ax.plot(psd.fr, psd.psd, color="k", label="Input")
for sigma in psd.sigma.values:
    ax.plot(
        psd.fr_hp, psd.psd_hp.sel(sigma=sigma), lw=0.9, label=rf"$\sigma = {sigma}$"
    )
ax.legend()

## Lambda

In [ ]:
swath = cases[0].load()
sigma_m = 300
psd_list = []
for lam in [0, 0.01, 0.1, 0.5, 1, 10]:
    psd = all_in_one(swath, lam=lam, sigma_m=sigma_m)
    psd["psd_hp"] = psd.psd_hp.expand_dims(lam=[lam])
    psd_list.append(psd)
psd = xr.concat(psd_list, "lam", data_vars=None)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 2.5), dpi=200)
fig.suptitle(rf"Varying λ ($\sigma = {sigma_m})$")

ax.set_xscale("log")
ax.set_yscale("log")
ax.plot(psd.fr, psd.psd, color="k", label="Input")
for lam in psd.lam.values:
    ax.plot(psd.fr_hp, psd.psd_hp.sel(lam=lam), lw=0.9, label=rf"$\lambda = {lam}$")
ax.legend()

## Multiple cases

In [ ]:
cases = get_cases()

In [ ]:
for case in cases:
    print(case.name)
    lam = 0.5
    psd_list = []
    for sigma in [130, 175, 198, 250, 300, 400]:
        psd = all_in_one(case.swath, lam=lam, sigma_m=sigma)
        psd["psd_hp"] = psd.psd_hp.expand_dims(sigma=[sigma])
        psd_list.append(psd)
    case.psd = xr.concat(psd_list, "sigma", data_vars=None)

In [ ]:
fig, axes = plt.subplots(
    1, 3, sharex=True, layout="constrained", figsize=(7, 2.5), dpi=200
)
fig.suptitle(rf"Varying σ ($\lambda = {lam})$")

for ax, case in zip(axes, cases):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.plot(case.psd.fr, case.psd.psd, color="k", label="Input")
    ax.plot(case.ef.frequency*1000, case.ef/1000, color="k", ls="--", label="WW3")
    for sigma in case.psd.sigma.values:
        ax.plot(
            case.psd.fr_hp,
            case.psd.psd_hp.sel(sigma=sigma),
            lw=0.9,
            label=rf"$\sigma = {sigma}$",
        )
    ax.set_title(rf"{case.name}")
    ax.set_xlim(1, case.psd.fr_hp[-1])
    ax.set_ylim(5e-6, 1)
axes[1].legend(ncols=2)

In [ ]:
for case in cases:
    print(case.name)
    sigma_m = 300
    psd_list = []
    for lam in [0, 0.01, 0.1, 0.5, 1, 10]:
        psd = all_in_one(case.swath, lam=lam, sigma_m=sigma_m)
        psd["psd_hp"] = psd.psd_hp.expand_dims(lam=[lam])
        psd_list.append(psd)
    case.psd = xr.concat(psd_list, "lam", data_vars=None)

In [ ]:
fig, axes = plt.subplots(
    1, 3, sharex=True, layout="constrained", figsize=(7, 2.5), dpi=200
)
fig.suptitle(rf"Varying λ ($\sigma = {sigma_m})$")

for ax, case in zip(axes, cases):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.plot(case.psd.fr, case.psd.psd, color="k", label="Input")
    for lam in case.psd.lam.values:
        ax.plot(
            case.psd.fr_hp,
            case.psd.psd_hp.sel(lam=lam),
            lw=0.9,
            label=rf"$\lambda = {lam}$",
        )
        ax.set_title(rf"{case.name}")
    ax.set_xlim(1, case.psd.fr_hp[-1])
    ax.set_ylim(None, 1)
axes[1].legend(ncols=2)

## Test effect pixel size

In [ ]:
n = 4096
dx = np.linspace(0.199, 0.200, n)
true_x = np.arange(n) * dx
x = np.arange(n) * np.mean(dx)

central = 10
y = np.sin(2*np.pi * true_x / central) + 0.001*np.random.normal(size=n)

plt.plot(x, y)

In [ ]:
fft = np.fft.fft(y, norm="ortho")
fft = np.fft.fftshift(fft)
fft = np.abs(fft) ** 2

f = np.fft.fftshift(np.fft.fftfreq(n, d=np.mean(dx)))
# f = np.fft.fftfreq(n, d=np.mean(dx))

fig, ax = plt.subplots()
ax.plot(f, fft)
ax.axvline(1/central)
ax.axvline(1/(central - 0.100))

In [ ]:
import finufft

f = np.linspace(0, 5, 1024)
fft = finufft.nufft1d3(true_x, y.astype("complex"), f)

fig, ax = plt.subplots()
ax.plot(f/(2*np.pi), np.abs(fft))
ax.axvline(1/central)

In [ ]:
N = 256
fft = finufft.nufft1d1(true_x, y.astype("complex"), N)

fig, ax = plt.subplots()
ax.plot(np.linspace(-2*np.pi, 2*np.pi, N), np.abs(fft))
ax.axvline(1/central)